# Equipe: VitaGuard Seguros

Membros:
* Lucas Lima Simões
* Raphael Hilário Alcova Coelho
* Patricia Tavares Simões
* Thayná dos Santos Patricio Furtado
* Willian Mello Antunes



# 🪛 Montagem do modelo

⚙️ Instalação e importação de bibliotecas

Nesta etapa, instalamos e importamos as bibliotecas necessárias para o funcionamento do sistema:

* transformers: para uso do modelo de linguagem (LLM)
* sentence-transformers: para geração de embeddings
* faiss: para busca vetorial eficiente (RAG)
* pypdf: para leitura de documentos PDF
* pandas e numpy: manipulação de dados
* random: geração de dados sintéticos

In [ ]:
# 🔹 Instalação das bibliotecas necessárias
!pip install -q transformers sentence-transformers faiss-cpu
!pip install -q pypdf

# 🔹 Importação das bibliotecas
import pandas as pd      # Manipulação de dados (DataFrame)
import numpy as np       # Operações numéricas
import faiss             # Busca vetorial (RAG)
import random            # Geração de dados sintéticos

# 🔹 Observação:
# Essas bibliotecas são a base do sistema:
# - FAISS → busca semântica
# - Sentence Transformers → embeddings
# - Transformers → LLM (geração de linguagem)

📊 Carregamento e geração de dados sintéticos

Como não utilizamos dados reais de uma seguradora, foi criada uma base de dados sintética com o objetivo de simular interações reais de clientes.

Os dados foram gerados com auxílio de IA e estruturados em formato de FAQ (pergunta → resposta → categoria), permitindo:

* Treinar o sistema de busca (RAG)
* Simular diferentes tipos de dúvidas de clientes
* Criar variações linguísticas para maior robustez

🧩 Criação do dataset de perguntas

Nesta etapa, definimos:

* Categorias de atendimento (pagamento, sinistro, cancelamento, etc.)
* Perguntas base para cada categoria
* Respostas padrão
* Variações linguísticas para gerar diversidade

O objetivo é simular como diferentes usuários fariam a mesma pergunta de formas distintas.

In [ ]:
# 🔹 Definição das categorias do sistema (incluindo novas categorias)
categorias = [
    "pagamento",
    "sinistro",
    "cancelamento",
    "cobertura",
    "carência",
    "beneficiário",
    "documentação",
    "atendimento",

    # 🔥 Categorias adicionais para enriquecer o chatbot
    "vencimento",
    "reajuste",
    "plano",
    "portabilidade",
    "renovacao",
    "dados_cadastrais",
    "fraude",
    "apolice",
    "carteirinha"
]

# 🔹 Perguntas base por categoria (simulam perguntas reais)
base_perguntas = {
    "pagamento": [
        "Como emitir segunda via do boleto?",
        "Como pagar meu seguro?",
        "Posso pagar após o vencimento?",
        "Onde encontro meu boleto?",
    ],
    "sinistro": [
        "Como acionar o seguro?",
        "Como abrir um sinistro?",
        "O que fazer em caso de falecimento?",
        "Como solicitar indenização?",
    ],
    "cancelamento": [
        "Como cancelar meu seguro?",
        "Posso cancelar a qualquer momento?",
        "Existe multa para cancelamento?",
    ],
    "cobertura": [
        "O que meu seguro cobre?",
        "Quais são as coberturas disponíveis?",
        "Seguro cobre morte natural?",
    ],
    "carência": [
        "Qual o período de carência?",
        "Quando posso usar o seguro?",
    ],
    "beneficiário": [
        "Como alterar beneficiário?",
        "Quem pode ser beneficiário?",
    ],
    "documentação": [
        "Quais documentos são necessários?",
        "Preciso enviar documentos para acionar?",
    ],
    "atendimento": [
        "Qual o telefone de atendimento?",
        "Como falar com um atendente?",
    ],
    "vencimento": [
        "Como alterar a data de vencimento do boleto?",
        "Posso mudar o vencimento da fatura?",
        "Como escolher outra data de pagamento?",
    ],
    "reajuste": [
        "Por que o valor do seguro aumentou?",
        "Como funciona o reajuste do seguro?",
        "O preço pode mudar com o tempo?",
    ],
    "plano": [
        "Como mudar meu plano?",
        "Posso fazer upgrade do seguro?",
        "Como alterar meu tipo de plano?",
    ],
    "portabilidade": [
        "Posso transferir meu seguro para outra seguradora?",
        "Como funciona portabilidade de seguro?",
    ],
    "renovacao": [
        "O seguro renova automaticamente?",
        "Como renovar minha apólice?",
    ],
    "dados_cadastrais": [
        "Como atualizar meus dados?",
        "Posso alterar meu endereço?",
        "Como mudar meu telefone cadastrado?",
    ],
    "fraude": [
        "O que fazer em caso de suspeita de fraude?",
        "Como denunciar fraude no seguro?",
    ],
    "apolice": [
        "Onde encontro minha apólice?",
        "Como acessar o contrato do seguro?",
    ],
    "carteirinha": [
        "Como obter minha carteirinha do seguro?",
        "Onde vejo meu número de cliente?",
    ]
}

# 🔹 Respostas padrão por categoria
base_respostas = {
    "pagamento": "Acesse o portal do cliente e clique em '2ª via de boleto'.",
    "sinistro": "Entre em contato pelo telefone 0800 ou utilize o aplicativo.",
    "cancelamento": "Você pode solicitar o cancelamento pelo portal ou atendimento.",
    "cobertura": "As coberturas variam conforme o plano contratado.",
    "carência": "O período de carência depende do tipo de cobertura.",
    "beneficiário": "A alteração pode ser feita pelo portal do cliente.",
    "documentação": "Os documentos variam conforme o tipo de solicitação.",
    "atendimento": "Entre em contato pelo telefone 0800 ou chat online.",
    "vencimento": "Você pode alterar a data de vencimento pelo portal do cliente na área financeira.",
    "reajuste": "O valor do seguro pode ser reajustado conforme condições contratuais.",
    "plano": "A alteração de plano pode ser solicitada pelo portal ou atendimento.",
    "portabilidade": "A portabilidade depende das regras da seguradora e análise do contrato.",
    "renovacao": "O seguro pode ser renovado automaticamente conforme o contrato.",
    "dados_cadastrais": "Você pode atualizar seus dados pelo portal do cliente.",
    "fraude": "Entre em contato imediatamente com o atendimento para reportar suspeita de fraude.",
    "apolice": "A apólice está disponível no portal do cliente na área de documentos.",
    "carteirinha": "Você pode acessar sua carteirinha digital pelo aplicativo ou portal."
}

# 🔹 Variações linguísticas (simulam diferentes formas de perguntar)
variacoes = [
    "Como faço para {}",
    "Quero saber {}",
    "Pode me dizer {}",
    "Preciso saber {}",
    "Como funciona {}",
    "Gostaria de entender {}",
    "Tem como explicar {}",
    "Onde vejo {}",
    "Como resolvo {}",
    "{}",
]

# 🔹 Geração dos dados sintéticos
dados = []

# 🔥 Aumenta volume e diversidade
for _ in range(2000):
    categoria = random.choice(categorias)
    pergunta_base = random.choice(base_perguntas[categoria])
    variacao = random.choice(variacoes)

    pergunta = variacao.format(pergunta_base.lower())

    dados.append({
        "pergunta": pergunta,
        "resposta": base_respostas[categoria],
        "categoria": categoria
    })

# 🔹 Cria DataFrame
df = pd.DataFrame(dados)

df.head()

,pergunta,resposta,categoria
0,Como funciona como escolher outra data de paga...,Você pode alterar a data de vencimento pelo po...,vencimento
1,Tem como explicar como emitir segunda via do b...,Acesse o portal do cliente e clique em '2ª via...,pagamento
2,Como funciona qual o telefone de atendimento?,Entre em contato pelo telefone 0800 ou chat on...,atendimento
3,Como faço para o seguro renova automaticamente?,O seguro pode ser renovado automaticamente con...,renovacao
4,Onde vejo como alterar meu tipo de plano?,A alteração de plano pode ser solicitada pelo ...,plano


💾 Exportação do dataset

Após a geração dos dados, o dataset é salvo em formato CSV.
Isso permite reutilização posterior sem necessidade de gerar novamente os dados.

In [ ]:
df.to_csv("vita_guard_dataset.csv", index=False)

🔄 Preparação dos dados para o modelo

Nesta etapa, carregamos o dataset e organizamos os dados em listas separadas:

* texts: perguntas (entrada do usuário)
* respostas: respostas associadas
* categorias: contexto das perguntas

Essas listas serão usadas na etapa de embeddings e busca vetorial.

In [ ]:
# 🔹 Carrega o dataset salvo
df = pd.read_csv("vita_guard_dataset.csv")

# 🔹 Separa colunas em listas
texts = df["pergunta"].tolist()
respostas = df["resposta"].tolist()
categorias = df["categoria"].tolist()

📄 Leitura de documentos PDF

Além dos dados estruturados (FAQ), o sistema também utiliza documentos não estruturados (PDFs), simulando materiais reais de uma seguradora:

* Contratos
* Apólices
* Termos e condições
* Guias de sinistro

Esses documentos enriquecem o contexto do chatbot.

In [ ]:
from pypdf import PdfReader
import os

# 🔹 Função para extrair texto dos PDFs
def extrair_texto_pdf(pasta="/content"):
    documentos = []

    for arquivo in os.listdir(pasta):
        if arquivo.endswith(".pdf"):
            caminho = os.path.join(pasta, arquivo)
            reader = PdfReader(caminho)

            texto = ""
            for pagina in reader.pages:
                conteudo = pagina.extract_text()
                if conteudo:
                    texto += conteudo + "\n"

            # 🔹 Guarda texto + fonte
            documentos.append({
                "fonte": arquivo,
                "texto": texto
            })

    return documentos

# 🔹 Executa leitura
documentos = extrair_texto_pdf()

print(f"{len(documentos)} PDFs carregados")

4 PDFs carregados


✂️ Processamento dos documentos (Chunking)

Os textos dos PDFs são divididos em pequenos trechos (chunks).

Isso é necessário porque:

* Modelos de embedding funcionam melhor com textos menores
* Permite buscas mais precisas no RAG
* Evita perda de contexto relevante

In [ ]:
# 🔹 Função para dividir texto em chunks
def dividir_chunks(texto, tamanho=300):
    palavras = texto.split()
    chunks = []

    for i in range(0, len(palavras), tamanho):
        chunk = " ".join(palavras[i:i+tamanho])
        chunks.append(chunk)

    return chunks

# 🔹 Gera chunks com referência à fonte
chunks_docs = []

for doc in documentos:
    partes = dividir_chunks(doc["texto"])

    for p in partes:
        chunks_docs.append({
            "texto": p,
            "fonte": doc["fonte"]
        })

print(f"{len(chunks_docs)} chunks gerados")

9 chunks gerados


🧠 Geração de embeddings e índice vetorial (RAG)

Nesta etapa, transformamos os textos em vetores numéricos (embeddings), que representam o significado semântico das frases.

Utilizamos:

* SentenceTransformer: para gerar embeddings das perguntas e dos documentos
* FAISS: para realizar buscas rápidas por similaridade

Fluxo:

1. Unimos os dados estruturados (FAQ) com os dados não estruturados (PDFs)
2. Geramos embeddings para todos os textos
3. Criamos um índice vetorial com FAISS

Isso permite que o chatbot encontre respostas com base em semelhança de significado, e não apenas por palavras exatas.

In [ ]:
# 🔹 Junta FAQ (perguntas) + documentos (chunks)
texts_total = texts + [c["texto"] for c in chunks_docs]

from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

# 🔹 Carrega modelo de embeddings
# Esse modelo transforma texto em vetores semânticos
model_emb = SentenceTransformer('all-MiniLM-L6-v2')

# 🔹 Gera embeddings para todos os textos
# (cada texto vira um vetor numérico)
embeddings = model_emb.encode(texts_total)

# 🔹 Cria índice vetorial FAISS
# Esse índice permite buscar textos similares rapidamente
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)

# 🔹 Adiciona os vetores ao índice
index.add(np.array(embeddings))

print("Embeddings e índice FAISS criados com sucesso!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings e índice FAISS criados com sucesso!


🔎 Recuperação de contexto (RAG)

Nesta etapa implementamos o mecanismo de RAG (Retrieval-Augmented Generation).

A função buscar_contexto é responsável por:

* Converter a pergunta do usuário em embedding
* Buscar os textos mais similares no índice FAISS
* Retornar os resultados mais relevantes

Os resultados podem vir de duas fontes:

* 📊 FAQ (dados estruturados) → respostas diretas
* 📄 Documentos (PDFs) → contexto mais detalhado

Essa separação permite que o chatbot combine precisão (FAQ) com profundidade (documentos).

In [ ]:
def buscar_contexto(pergunta, k=3):
    """
    🔎 Função de busca semântica (RAG)

    Parâmetros:
    - pergunta: texto do usuário
    - k: número de resultados mais próximos

    Retorna:
    - lista de resultados relevantes (FAQ + documentos)
    """

    # 🔹 Converte pergunta em embedding
    emb = model_emb.encode([pergunta])

    # 🔹 Busca os k vetores mais próximos no FAISS
    D, I = index.search(np.array(emb), k)

    resultados = []

    # 🔹 Percorre resultados encontrados
    for i, dist in zip(I[0], D[0]):

        # 🔸 Caso 1: resultado pertence ao FAQ
        if i < len(respostas):
            resultados.append({
                "resposta": respostas[i],
                "categoria": categorias[i],
                "tipo": "faq",
                "score": float(dist)  # menor = mais similar
            })

        # 🔸 Caso 2: resultado pertence aos documentos (PDFs)
        else:
            idx_doc = i - len(respostas)

            resultados.append({
                "resposta": chunks_docs[idx_doc]["texto"],
                "categoria": "documento",
                "tipo": "doc",
                "fonte": chunks_docs[idx_doc]["fonte"],
                "score": float(dist)
            })

    return resultados

✅ Validação de resposta (controle de confiança)

Nesta etapa, implementamos um mecanismo de validação de resposta, que evita que o chatbot responda quando não há confiança suficiente.

Como funciona:

* O FAISS retorna os resultados mais próximos com um score de distância
* Quanto menor o score → maior a similaridade
* Se o score for maior que um limite (threshold), consideramos a resposta não confiável

Isso permite que o chatbot:

* Evite “alucinações”
* Responda apenas quando há segurança
* Retorne fallback como “não sei responder” quando necessário

In [ ]:
def resposta_confiavel(resultados, threshold=1.0):
    """
    ✅ Verifica se a resposta encontrada é confiável

    Parâmetros:
    - resultados: lista retornada pelo RAG
    - threshold: limite máximo de distância aceitável

    Retorna:
    - resposta (str) se confiável
    - None se não confiável
    """

    # 🔹 Garante que há resultados
    if not resultados:
        return None

    # 🔹 Pega o melhor resultado (menor distância)
    melhor = resultados[0]

    # 🔹 Se a distância for alta, não confiamos
    if melhor["score"] > threshold:
        return None

    # 🔹 Caso contrário, retorna a resposta
    return melhor["resposta"]

🧠 Memória de conversa

Para tornar a interação mais natural, implementamos uma memória de curto prazo do chatbot.

Essa memória permite:

* Lembrar perguntas recentes do usuário
* Identificar repetição de tópicos
* Melhorar a continuidade da conversa

Cada interação armazenada contém:

* Pergunta do usuário
* Resposta fornecida
* Tópicos identificados
* Embedding da pergunta

Além disso, limitamos o histórico aos últimos 5 registros para evitar crescimento excessivo e manter apenas o contexto recente.

In [ ]:
# 🔹 Lista que armazena o histórico recente da conversa
historico = []

def atualizar_memoria(pergunta, resposta, topicos):
    """
    🧠 Atualiza a memória da conversa

    Parâmetros:
    - pergunta: pergunta do usuário
    - resposta: resposta gerada pelo sistema
    - topicos: categorias associadas à pergunta

    A função:
    - Gera embedding da pergunta
    - Armazena no histórico
    - Mantém apenas os últimos 5 registros
    """

    # 🔹 Gera embedding da pergunta (usado para similaridade futura)
    emb = model_emb.encode([pergunta])[0]

    # 🔹 Adiciona ao histórico
    historico.append({
        "pergunta": pergunta.lower(),
        "resposta": resposta,
        "topicos": topicos,
        "embedding": emb
    })

    # 🔹 Mantém apenas os últimos 5 itens (memória curta)
    if len(historico) > 5:
        historico.pop(0)

🤖 Uso do modelo de linguagem (LLM)

Nesta etapa, utilizamos um modelo de linguagem (LLM) como parte da arquitetura do chatbot.

Embora a resposta final seja construída principalmente a partir do mecanismo de RAG (Retrieval-Augmented Generation), o LLM é utilizado para tarefas de processamento de linguagem natural, especialmente:

classificação de intenção do usuário;
apoio ao entendimento do contexto da pergunta;
demonstração prática do uso de IA generativa no projeto.

🔗 Como o LLM é utilizado no projeto

O fluxo geral funciona assim:

1. O usuário envia uma pergunta;
2. O sistema identifica a intenção da mensagem (com apoio do LLM);
3. O RAG busca a melhor resposta nos dados (FAQ ou documentos);
4. A resposta é validada;
5. Uma camada de humanização programada torna a resposta mais natural;
6. A memória da conversa é atualizada.

👉 Ou seja, o LLM não gera diretamente a resposta final, mas participa da inteligência do sistema, especialmente na etapa de interpretação.

⚙️ Escolha do modelo

Utilizamos o modelo FLAN-T5 Small, por alguns motivos importantes:

* ✅ Gratuito e open-source
* ✅ Leve (ideal para rodar no Google Colab)
* ✅ Treinado para seguir instruções (instruction-tuned)
* ✅ Adequado para classificação e tarefas de NLP

Embora existam modelos maiores (como T5-base ou LLaMA), eles exigem mais recursos computacionais e não eram necessários para este projeto.

📌 Limitação

Por ser um modelo pequeno, o FLAN-T5 pode:

* apresentar menor precisão em classificações complexas;
* eventualmente gerar interpretações menos robustas.

Mesmo assim, ele é suficiente para demonstrar o uso de LLM dentro da arquitetura proposta.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 🔹 Modelo escolhido (leve e eficiente)
model_name = "google/flan-t5-small"

# 🔹 Tokenizer: transforma texto em tokens (entrada do modelo)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 🔹 Modelo LLM: responsável por gerar/reformular texto
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 🔹 Observação:
# Esse modelo será usado principalmente para detecção de intenção,
# auxiliando o chatbot na interpretação da pergunta do usuário.

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


🎯 Detecção de intenção

Nesta etapa, implementamos um mecanismo de detecção de intenção, que permite ao chatbot entender o objetivo da pergunta do usuário.

Isso é importante para:

* Adaptar o tipo de resposta
* Identificar perguntas sobre histórico
* Diferenciar tipos de solicitação (pagamento, sinistro, etc.)

🔗 Estratégia híbrida (Regras + LLM)

Utilizamos uma abordagem híbrida:

🔥 1. Regras (alta prioridade)

Para casos críticos (como perguntas sobre histórico), utilizamos palavras-chave:

* “já perguntei”
* “já falei”
* “já mencionei”

👉 Isso garante respostas rápidas e precisas nesses casos.

🤖 2. LLM (fallback)

Caso nenhuma regra seja acionada, usamos o modelo de linguagem para classificar a intenção.

O modelo recebe um prompt estruturado e retorna uma categoria.

📌 Vantagens dessa abordagem
* Mais confiável que usar apenas LLM
* Mais flexível que usar apenas regras
* Evita erros em casos importantes

In [ ]:
def detectar_intencao(pergunta):
    """
    🎯 Detecta a intenção da pergunta do usuário

    Estratégia:
    1. Regras (alta prioridade)
    2. LLM (fallback)

    Retorna:
    - uma das categorias do dataset
    - "historico" (quando a pergunta se refere ao histórico)
    - "geral" (fallback)
    """

    pergunta_lower = pergunta.lower()

    # 🔥 REGRA FORTE
    gatilhos_historico = [
        "já perguntei", "ja perguntei",
        "já falei", "ja falei",
        "já mencionei", "ja mencionei"
    ]

    if any(g in pergunta_lower for g in gatilhos_historico):
        return "historico"

    # 🔹 Lista completa de categorias
    categorias_validas = [
        "pagamento", "sinistro", "cancelamento", "cobertura",
        "carência", "beneficiário", "documentação", "atendimento",
        "vencimento", "reajuste", "plano", "portabilidade",
        "renovacao", "dados_cadastrais", "fraude", "apolice", "carteirinha"
    ]

    # 🔹 Prompt dinâmico
    opcoes = "\n".join([f"- {c}" for c in categorias_validas])

    prompt = f"""
Classifique a intenção da pergunta abaixo em UMA palavra.

Opções:
{opcoes}
- historico
- geral

Pergunta: {pergunta}

Resposta:
"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=128)

    outputs = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False
    )

    resposta = tokenizer.decode(outputs[0], skip_special_tokens=True).lower().strip()

    # 🔹 Normalização robusta
    for cat in categorias_validas:
        if cat in resposta:
            return cat

    if "histor" in resposta:
        return "historico"

    return "geral"

🔁 Detecção de perguntas repetidas

Para tornar a conversa mais natural, implementamos um mecanismo que identifica quando o usuário faz uma pergunta semelhante a outra já realizada anteriormente.

Essa funcionalidade permite:

* Evitar respostas redundantes
* Criar sensação de memória real
* Melhorar a experiência conversacional

🔗 Estratégia utilizada

A detecção de repetição combina duas abordagens:

🔹 1. Similaridade por tópico (mais forte)

Se a nova pergunta contém palavras-chave associadas a tópicos já discutidos, consideramos como repetição.

🔹 2. Similaridade semântica (embeddings)

Utilizamos similaridade de cosseno entre embeddings para identificar perguntas semanticamente parecidas, mesmo com palavras diferentes.

📌 Vantagem

Essa abordagem permite detectar equivalência entre frases como:

* “Como emitir boleto?”
* “Onde tiro a segunda via?”

In [ ]:
def verificar_repeticao(pergunta, threshold=0.6):
    """
    🔁 Detecta se a pergunta já foi feita anteriormente

    Estratégia:
    1. Match por palavras-chave/tópicos
    2. Similaridade semântica (cosine similarity)

    Parâmetros:
    - pergunta: pergunta atual do usuário
    - threshold: limite de similaridade

    Retorna:
    - resposta anterior (se repetida)
    - None (se não for repetida)
    """

    # 🔹 Se não há histórico, não há repetição
    if len(historico) == 0:
        return None

    pergunta_lower = pergunta.lower()

    # 🔹 Extrai palavras-chave (ignora palavras muito curtas)
    palavras = [p for p in pergunta_lower.split() if len(p) > 3]

    # 🔹 Embedding da pergunta atual
    emb_pergunta = model_emb.encode([pergunta])[0]

    melhor_score = 0
    melhor_resposta = None

    for item in historico:

        # 🔥 1. MATCH POR TÓPICO (prioridade alta)
        # verifica se alguma palavra-chave aparece nos tópicos
        if any(p in item["topicos"] for p in palavras):
            return item["resposta"]

        # 🔹 2. SIMILARIDADE SEMÂNTICA (cosine similarity)
        emb_hist = item["embedding"]

        score = np.dot(emb_pergunta, emb_hist) / (
            np.linalg.norm(emb_pergunta) * np.linalg.norm(emb_hist)
        )

        # 🔹 Guarda melhor resultado
        if score > melhor_score:
            melhor_score = score
            melhor_resposta = item["resposta"]

    # 🔹 Se similaridade for alta, considera repetição
    if melhor_score > threshold:
        return melhor_resposta

    return None

🤖 Função principal do chatbot

Nesta etapa, integramos todos os componentes desenvolvidos:

* Detecção de intenção
* Busca semântica (RAG)
* Validação de resposta
* Memória de conversa
* Humanização da resposta

A função responder representa o fluxo completo do chatbot.

🔗 Fluxo de execução
1. Identifica a intenção da pergunta
2. Verifica se é uma pergunta sobre histórico
3. Busca contexto relevante com RAG
4. Valida se a resposta é confiável
5. Aplica humanização programada na resposta
6. Atualiza a memória da conversa

📌 Objetivo

Garantir que o chatbot seja:

* Natural
* Contextual
* Consistente
* Robusto

In [ ]:
# 🔹 Identidade do chatbot
NOME_BOT = "Vita"
EMPRESA = "VitaGuard Seguros"

import random

# 🔹 Remove prefixos repetitivos (limpeza de resposta)
def limpar_resposta(texto):
    remover = [
        "Claro! ",
        "Sem problema! ",
        "Aqui vai a informação: ",
        "Posso te ajudar com isso! ",
        "Vamos lá! "
    ]

    for r in remover:
        texto = texto.replace(r, "")

    return texto.strip()


# 🔹 Adiciona humanização à resposta
def tornar_mais_humano(resposta):
    prefixos = [
        "Claro!",
        "Sem problema!",
        "Posso te ajudar com isso!",
        "Vamos lá!",
        "Aqui vai a informação:"
    ]

    return f"{random.choice(prefixos)} {resposta}"


# 🔹 Função principal do chatbot
def responder(pergunta):
    """
    🤖 Pipeline principal do chatbot

    Etapas:
    - Detecta intenção
    - Consulta histórico (se necessário)
    - Busca resposta via RAG
    - Valida resposta
    - Humaniza resposta
    - Atualiza memória
    """

    # 🔹 1. Detecta intenção
    intencao = detectar_intencao(pergunta)

    # 🔹 2. Tratamento de histórico
    if intencao == "historico":
        resposta_antiga = verificar_repeticao(pergunta)

        if resposta_antiga:
            resposta_antiga = limpar_resposta(resposta_antiga)
            return f"Sim 😊 você já perguntou isso. Mas respondendo novamente: {resposta_antiga}"
        else:
            return "Não encontrei essa pergunta no nosso histórico recente."

    # 🔹 3. Busca contexto com RAG
    resultados = buscar_contexto(pergunta)

    # 🔹 4. Validação de confiança
    resposta_base = resposta_confiavel(resultados)

    if not resposta_base:
        return "Desculpe, não encontrei essa informação. Pode reformular?"

    # 🔹 5. Identificação de múltiplos tópicos (top 2 resultados)
    topicos = list(set([r["categoria"] for r in resultados[:2]]))

    # 🔹 6. Limpeza antes de humanizar (evita eco)
    resposta_base = limpar_resposta(resposta_base)

    # 🔹 7. Humanização (sem uso de LLM)
    resposta_final = tornar_mais_humano(resposta_base)

    # 🔹 8. Atualiza memória (inclui tópicos)
    atualizar_memoria(pergunta, resposta_final, " ".join(topicos).lower())

    return resposta_final

# 💬 Interface de interação com o usuário

Nesta etapa final, implementamos um loop de interação simples via terminal, permitindo conversar com o chatbot em tempo real.

O usuário pode:

* Fazer perguntas livremente
* Receber respostas contextualizadas
* Encerrar a conversa a qualquer momento

📌 Observações
* O chatbot mantém memória durante a sessão
* O comando "sair" ou "exit" ou "quit" encerra a conversa
* As respostas são geradas dinamicamente pelo pipeline completo

In [ ]:
# 🔹 Mensagem inicial
print(f"Olá! Eu sou a {NOME_BOT}, assistente da {EMPRESA}. Como posso ajudar você hoje?\n")

while True:
    try:
        pergunta = input("Você: ").strip()

        # 🔹 Ignora entrada vazia
        if not pergunta:
            print(f"{NOME_BOT}: Poderia me dizer como posso ajudar?")
            continue

        # 🔹 Comando de saída
        if pergunta.lower() in ["sair", "exit", "quit"]:
            print(f"{NOME_BOT}: Foi um prazer ajudar! Até mais 😊")
            break

        # 🔹 Gera resposta
        resposta = responder(pergunta)

        print(f"{NOME_BOT}: {resposta}")

    except KeyboardInterrupt:
        print(f"\n{NOME_BOT}: Conversa encerrada. Até mais 😊")
        break

    except Exception as e:
        print(f"{NOME_BOT}: Ocorreu um erro ao processar sua solicitação. Tente novamente.")
        print("Erro técnico:", e)

Olá! Eu sou a Vita, assistente da VitaGuard Seguros. Como posso ajudar você hoje?

Você: Como posso emitir a segunda via do boleto?
Vita: Posso te ajudar com isso! Acesse o portal do cliente e clique em '2ª via de boleto'.
Você: E se que quiser alterar a data de vencimento?
Vita: Posso te ajudar com isso! Você pode alterar a data de vencimento pelo portal do cliente na área financeira.
Você: O que faço em caso de sinistro?
Vita: Aqui vai a informação: Entre em contato pelo telefone 0800 ou utilize o aplicativo.
Você: Já perguntei do boleto?
Vita: Sim 😊 você já perguntou isso. Mas respondendo novamente: Acesse o portal do cliente e clique em '2ª via de boleto'.
Você: Quero alterar meus dados
Vita: Vamos lá! Você pode atualizar seus dados pelo portal do cliente.
Você: Detectei fraude
Vita: Vamos lá! Entre em contato imediatamente com o atendimento para reportar suspeita de fraude.
Você: sair
Vita: Foi um prazer ajudar! Até mais 😊
